In [1]:
from os import getenv
from typing import ClassVar, Dict, List, Optional

from dotenv import load_dotenv
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

from pydantic.dataclasses import dataclass
import pytidycensus as tc


load_dotenv()


True

# RUCA Loader

# Census Loader

In [2]:
class CensusDataLoader(BaseModel):


    NON_CONTINENTAL: ClassVar[Set[str]] = {
        "Alaska", "Hawaii", "American Samoa", "Guam",
        "Commonwealth of the Northern Mariana Islands", "Puerto Rico",
        "United States Virgin Islands",
    }
    
    year: int = Field(default=2024, ge=2010, le=2026)
    states: Optional[List[str]] = None
    api_key: Optional[str] = None



    continental: bool = True

    def load_ruca(self) -> pd.DataFrame:
        """Returns a cleaned RUCA DataFrame, optionally filtered to continental US."""

        URL: str = "https://www.ers.usda.gov/media/5443/2020-rural-urban-commuting-area-codes-census-tracts.csv?v=48133"
        
        cols_to_load = [
            "TractFIPS20", "CountyFIPS20", "CountyName20",
            "StateFIPS20", "StateName20", "PrimaryRUCA",
        ]
        dtypes = {
            "TractFIPS20": str,
            "CountyFIPS20": str,
            "StateFIPS20": str,
            "StateName20": "category",
            "CountyName20": "category",
            "PrimaryRUCA": "int8",
        }
        ruca: pd.DataFrame = pd.read_csv(
            URL,
            encoding="latin1",
            usecols=cols_to_load,
            dtype=dtypes,
        )
        ruca.columns = ruca.columns.str.replace("20$", "", regex=True)
        ruca.rename(columns={"PrimaryRUCA": "RUCA"}, inplace=True)

        if self.continental:
            return ruca[~ruca["StateName"].isin(self.NON_CONTINENTAL)].reset_index(drop=True)

        return ruca

    
    def model_post_init(self, __context) -> None:
        if self.api_key:
            try:
                tc.set_census_api_key(self.api_key)
            except ValueError:
                pass

    def fetch_acs(self, variables: List[str]) -> pd.DataFrame:
        """Fetches census variables using config states and year."""
        deduped_vars = sorted(list(set(variables)))
        return tc.get_acs(
            geography="tract",
            variables=deduped_vars,
            state=list(set(self.states)),
            year=self.year
        )

x = CensusDataLoader()
x.load_ruca()

,TractFIPS,CountyFIPS,CountyName,StateFIPS,StateName,RUCA
0,01001020100,01001,Autauga County,01,Alabama,1
1,01001020200,01001,Autauga County,01,Alabama,1
2,01001020300,01001,Autauga County,01,Alabama,1
3,01001020400,01001,Autauga County,01,Alabama,1
4,01001020501,01001,Autauga County,01,Alabama,1
...,...,...,...,...,...,...
83771,56043000200,56043,Washakie County,56,Wyoming,8
83772,56043000301,56043,Washakie County,56,Wyoming,7
83773,56043000302,56043,Washakie County,56,Wyoming,7
83774,56045951100,56045,Weston County,56,Wyoming,10


In [3]:
rent_burden_vars = [
    "B25070_001E",  # Total (Renter)
    "B25070_007E",  # Gross rent: 30.0 to 34.9 percent
    "B25070_008E",  # Gross rent: 35.0 to 39.9 percent
    "B25070_009E",  # Gross rent: 40.0 to 49.9 percent
    "B25070_010E",  # Gross rent: 50.0 percent or more
]

owner_burden_vars = [
    "B25091_001E",  # Total (Owner)
    "B25091_008E",  # Housing units with a mortgage: 30.0 to 34.9 percent
    "B25091_009E",  # Housing units with a mortgage: 35.0 to 39.9 percent
    "B25091_010E",  # Housing units with a mortgage: 40.0 to 49.9 percent
    "B25091_011E",  # Housing units with a mortgage: 50.0 percent or more
    "B25091_019E",  # Housing units without a mortgage: 30.0 to 34.9 percent
    "B25091_020E",  # Housing units without a mortgage: 35.0 to 39.9 percent
    "B25091_021E",  # Housing units without a mortgage: 40.0 to 49.9 percent
    "B25091_022E",  # Housing units without a mortgage: 50.0 percent or more
]

poverty_vars = [
    "B17001_001E",  # Total
    "B17001_002E",  # Income in the past 12 months below poverty level
]

housing_quality_vars = [
    "B25047_001E",  # Total units (Plumbing)
    "B25047_003E",  # Lacking complete plumbing facilities
    "B25051_001E",  # Total units (Kitchen)
    "B25051_003E",  # Lacking complete kitchen facilities
]

education_vars = [
    "B15003_001E",  # Total (Population 25 years and over)
] + [f"B15003_{num:03d}E" for num in range(2, 17)]

demographics_vars = [
    "B03002_001E",  # Total
    "B03002_003E",  # Not Hispanic or Latino: White alone
]

misc_vars = {
    "Gini": "B19083_001E",
}


In [4]:
x = CensusDataLoader()
x.load_ruca()

,TractFIPS,CountyFIPS,CountyName,StateFIPS,StateName,RUCA
0,01001020100,01001,Autauga County,01,Alabama,1
1,01001020200,01001,Autauga County,01,Alabama,1
2,01001020300,01001,Autauga County,01,Alabama,1
3,01001020400,01001,Autauga County,01,Alabama,1
4,01001020501,01001,Autauga County,01,Alabama,1
...,...,...,...,...,...,...
83771,56043000200,56043,Washakie County,56,Wyoming,8
83772,56043000301,56043,Washakie County,56,Wyoming,7
83773,56043000302,56043,Washakie County,56,Wyoming,7
83774,56045951100,56045,Weston County,56,Wyoming,10
